# Prompted Segmentation for Drywall QA

Fine-tune CLIPSeg for text-conditioned binary segmentation of drywall defects.

**Tasks:** Crack detection (`"segment crack"`) and taping area detection (`"segment taping area"`).

**Runtime:** Google Colab with GPU (T4 recommended).

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch torchvision transformers>=4.35 roboflow Pillow opencv-python numpy matplotlib scikit-learn pycocotools tqdm

In [ ]:
# (Optional) Mount Google Drive to save checkpoints
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Clone / Upload Project

In [ ]:
# Option A: Clone from GitHub (update URL)
# !git clone https://github.com/YOUR_USERNAME/drywall.git
# %cd drywall

# Option B: If you uploaded a zip
# !unzip drywall.zip
# %cd drywall

# Option C: Already in the right directory
import os
print(f"Working directory: {os.getcwd()}")
!ls -la

## 3. API Key

In [ ]:
# Paste your Roboflow API key here
import os
os.environ["ROBOFLOW_API_KEY"] = ""  # <-- paste your key between the quotes

## 4. Download Data

In [ ]:
!python data/download.py

## 5. Prepare Binary Masks

In [ ]:
!python data/prepare.py

## 6. Sanity Check — Display Sample Images + Masks

In [ ]:
import random
import matplotlib.pyplot as plt
import torch
from data.dataset import DrywallSegDataset

dataset = DrywallSegDataset(split="train")
indices = random.sample(range(len(dataset)), min(5, len(dataset)))

fig, axes = plt.subplots(len(indices), 2, figsize=(10, 4 * len(indices)))
mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)

for i, idx in enumerate(indices):
    img, prompt, mask, meta = dataset[idx]
    img_display = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    
    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f'{meta["task_type"]} — "{prompt}"')
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(mask.numpy(), cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()
print(f"Total training samples: {len(dataset)}")

## 7. Train

In [ ]:
!python train.py

## 8. Evaluate

In [ ]:
!python evaluate.py

## 9. Predict — Generate Output Masks

In [ ]:
!python predict.py

## 10. Visualize Results

In [ ]:
!python visualize.py

In [ ]:
# Display generated visualizations inline
import glob
from IPython.display import display, Image as IPImage

for img_path in sorted(glob.glob("outputs/visuals/*.png")):
    print(f"\n{os.path.basename(img_path)}")
    display(IPImage(filename=img_path, width=900))

## 11. Cross-Prompt Test (Text Conditioning Verification)

Run crack prompt on taping images (and vice versa). A well-conditioned model should produce near-empty masks.

In [ ]:
from model.clipseg_finetune import CLIPSegFinetune
from config import DEVICE, CHECKPOINT_DIR, CANONICAL_PROMPTS, THRESHOLD
import numpy as np

model = CLIPSegFinetune().to(DEVICE)
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"), map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Test: crack prompt on taping images
taping_dataset = DrywallSegDataset(split="test", task_filter="taping")
if len(taping_dataset) > 0:
    fg_ratios = []
    for idx in range(min(10, len(taping_dataset))):
        img, _, _, _ = taping_dataset[idx]
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(DEVICE), ["segment crack"])
            pred = torch.sigmoid(logits).squeeze().cpu().numpy()
        fg_ratios.append((pred > THRESHOLD).mean())
    print(f'"segment crack" on taping images → avg fg ratio: {np.mean(fg_ratios):.4f} (should be ~0)')

# Test: taping prompt on crack images
crack_dataset = DrywallSegDataset(split="test", task_filter="crack")
if len(crack_dataset) > 0:
    fg_ratios = []
    for idx in range(min(10, len(crack_dataset))):
        img, _, _, _ = crack_dataset[idx]
        with torch.no_grad():
            logits = model(img.unsqueeze(0).to(DEVICE), ["segment taping area"])
            pred = torch.sigmoid(logits).squeeze().cpu().numpy()
        fg_ratios.append((pred > THRESHOLD).mean())
    print(f'"segment taping area" on crack images → avg fg ratio: {np.mean(fg_ratios):.4f} (should be ~0)')

## 12. Download Results

In [ ]:
import shutil

# Zip outputs and checkpoints for download
shutil.make_archive("drywall_results", "zip", ".", "outputs")
shutil.make_archive("drywall_checkpoints", "zip", ".", "checkpoints")

print("Download these files:")
print("  - drywall_results.zip (masks + visualizations)")
print("  - drywall_checkpoints.zip (model weights)")

# On Colab, trigger download
try:
    from google.colab import files
    files.download("drywall_results.zip")
    files.download("drywall_checkpoints.zip")
except ImportError:
    pass